# PR Process Metrics — Journal Notebook

Este notebook melhora as análises de processo iniciadas em `prs_viz_bp_analysis.ipynb`.

## Métricas cobertas

| Seção | Métrica | Melhoria principal |
|---|---|---|
| 1 | Lead Time to Merge | Violin (log scale) · todos os 11 repos · versão filtrada |
| 2 | PR Complexity | Violin para comments e commits (em vez de barras de média) |
| 3 | Review Process | Violin para intensidade + bar para cobertura · todos os repos |
| 4 | Collective Ownership | Violin para distribuição de autores/arquivo · todos os repos |
| 5 | Statistical Tests | Kruskal-Wallis + Mann-Whitney pairwise com effect size |

## Por que distribuições em vez de médias?

O notebook original usava barras de média para comments, commits e reviewers.
Esse design perde toda a informação sobre variabilidade, assimetria e outliers:
dois projetos com a mesma média podem ter padrões radicalmente diferentes
(e.g., um consistente com PRs de 1–2 commits, outro com poucos PRs enormes puxando a média).
Violin plots expõem exatamente essas diferenças, que são relevantes para interpretar
maturidade de processo de desenvolvimento.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats

sns.set_context('paper', font_scale=1.25)
sns.set_style('whitegrid')
plt.rcParams.update({
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.constrained_layout.use': True,
})

GROUP_COLORS = {
    'Brasil Participativo': '#0B3C5D',
    'MDS (Academic)':       '#1B998B',
    'REQ (Academic)':       '#2DC7B4',
    'EPS (Academic)':       '#7EE8A2',
    'Decidim (Market)':     '#E07B14',
    'VSCode (Market)':      '#D55E00',
    'React (Market)':       '#F0AB44',
    'Storm (Market)':       '#A0522D',
    'Phoenix (Market)':     '#9B59B6',
    'PulpCore (Market)':    '#C0392B',
    'Quay (Market)':        '#E74C3C',
}

BP_PRIMARY   = '#0B3C5D'
ACAD_PRIMARY = '#1B998B'
MKT_PRIMARY  = '#E07B14'

DATA_PATH   = Path('data') / 'silver' / 'prs.csv'
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

print(f'Data: {DATA_PATH.resolve()}')

In [ ]:
BOT_KEYWORDS = [
    'bot', 'dependabot', 'renovate', 'github-actions', 'codecov',
    'greenkeeper', 'snyk', 'pyup', 'automated', 'ci-', 'action',
    'github-advanced-security', 'copilot-pull-request',
]

def is_bot(name: str) -> bool:
    if pd.isna(name):
        return False
    return any(kw in str(name).lower() for kw in BOT_KEYWORDS)


def count_human_reviewers(rv_str: str) -> int:
    """Count non-bot reviewers in a comma-separated reviewers string."""
    if pd.isna(rv_str) or str(rv_str).strip() == '':
        return 0
    return sum(1 for r in str(rv_str).split(',') if r.strip() and not is_bot(r.strip()))


df_raw = pd.read_csv(DATA_PATH)

GROUP_DEFS = {
    'Brasil Participativo': lambda d: (d['org'] == 'lappis-unb/decidimbr') & (d['repo'] == 'decidim-govbr'),
    'MDS (Academic)':       lambda d: d['org'] == 'unb-mds',
    'REQ (Academic)':       lambda d: d['org'] == 'mdsreq-fga-unb',
    'EPS (Academic)':       lambda d: d['org'] == 'fga-eps-mds',
    'Decidim (Market)':     lambda d: d['org'] == 'decidim',
    'VSCode (Market)':      lambda d: (d['org'] == 'microsoft') & (d['repo'] == 'vscode'),
    'React (Market)':       lambda d: (d['org'] == 'facebook') & (d['repo'] == 'react'),
    'Storm (Market)':       lambda d: (d['org'] == 'apache') & (d['repo'] == 'storm'),
    'Phoenix (Market)':     lambda d: (d['org'] == 'apache') & (d['repo'] == 'phoenix'),
    'PulpCore (Market)':    lambda d: (d['org'] == 'pulp') & (d['repo'] == 'pulpcore'),
    'Quay (Market)':        lambda d: (d['org'] == 'quay') & (d['repo'] == 'quay'),
}

groups: dict[str, pd.DataFrame] = {}
for label, mask_fn in GROUP_DEFS.items():
    subset = df_raw[mask_fn(df_raw)].copy()
    groups[label] = subset[~subset['author'].apply(is_bot)].copy()

print('Groups loaded:')
for k, v in groups.items():
    print(f'  {k:<30} {len(v):>6} PRs')

In [ ]:
def build_long(groups: dict, metric: str, min_val: float = 0,
               transform=None) -> pd.DataFrame:
    """Build a long-format DataFrame for violin plotting."""
    frames = []
    for label, df in groups.items():
        col = df[metric].dropna()
        col = col[col > min_val]
        if transform:
            col = col.apply(transform)
        frames.append(pd.DataFrame({'group': label, 'value': col.values}))
    return pd.concat(frames, ignore_index=True)


def violin(
    groups: dict,
    metric: str,
    ylabel: str,
    title: str,
    log_scale: bool = False,
    clip_pct: float = 99,
    min_val: float = 0,
    figsize: tuple = (12, 5),
    save_path: Path | None = None,
    custom_groups: dict | None = None,
) -> plt.Figure:
    """
    Generic publication-quality violin plot.
    custom_groups : pass a pre-filtered dict instead of the global groups.
    """
    src = custom_groups if custom_groups is not None else groups
    long = build_long(src, metric, min_val=min_val)

    # Clip per group for display
    p_clip  = long.groupby('group')['value'].quantile(clip_pct / 100)
    long['disp'] = long.apply(lambda r: min(r['value'], p_clip[r['group']]), axis=1)

    # Sort by median
    order = (
        long.groupby('group')['value']
        .median().sort_values().index.tolist()
    )
    palette = [GROUP_COLORS[g] for g in order]

    if log_scale:
        long['plot'] = np.log10(long['disp'])
    else:
        long['plot'] = long['disp']

    fig, ax = plt.subplots(figsize=figsize)

    sns.violinplot(
        data=long, x='group', y='plot',
        order=order, palette=palette,
        inner='box', bw_adjust=0.8, cut=0, linewidth=0.8, ax=ax,
    )

    # Median annotations (original scale)
    ylim = ax.get_ylim()
    for i, grp in enumerate(order):
        med = long.loc[long['group'] == grp, 'value'].median()
        fmt = f'{med:.1f}h' if 'hour' in metric.lower() or 'time' in metric.lower() else f'{med:.1f}'
        ax.text(i, ylim[1] * 0.97, fmt,
                ha='center', va='top', fontsize=8, color='#333',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.75, ec='none'))

    if log_scale:
        yticks = ax.get_yticks()
        ax.set_yticks(yticks)
        ax.set_yticklabels([f'{10**t:,.1f}' for t in yticks])
        ax.set_ylabel(f'{ylabel} (log scale)', fontweight='bold')
    else:
        ax.set_ylabel(ylabel, fontweight='bold')

    ax.set_xlabel('')
    ax.set_title(title, fontweight='bold', pad=10)
    ax.set_xticklabels(order, rotation=30, ha='right', fontsize=9)

    # Category bars below x-axis
    ylim2 = ax.get_ylim()
    bar_y  = ylim2[0] - (ylim2[1] - ylim2[0]) * 0.05
    n      = len(order)
    for idxs, color in [
        ([i for i, g in enumerate(order) if 'Participativo' in g], BP_PRIMARY),
        ([i for i, g in enumerate(order) if 'Academic'     in g], ACAD_PRIMARY),
        ([i for i, g in enumerate(order) if 'Market'       in g], MKT_PRIMARY),
    ]:
        if not idxs:
            continue
        ax.axhline(bar_y, xmin=(min(idxs)+0.05)/n, xmax=(max(idxs)+0.95)/n,
                   color=color, linewidth=4, solid_capstyle='butt', clip_on=False)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    handles = [
        mpatches.Patch(color=BP_PRIMARY,   label='Government (BP)'),
        mpatches.Patch(color=ACAD_PRIMARY, label='Academic'),
        mpatches.Patch(color=MKT_PRIMARY,  label='Market'),
    ]
    ax.legend(handles=handles, loc='upper left', frameon=True, framealpha=0.85,
              title='Category', title_fontsize=9)

    note = f'Up to {clip_pct}th percentile per group. Annotation = median.'
    ax.text(0.5, -0.22, note, transform=ax.transAxes,
            ha='center', fontsize=8, color='gray', style='italic')

    if save_path:
        fig.savefig(save_path, bbox_inches='tight')
        print(f'Saved: {save_path}')

    plt.show()
    return fig

---
## 1  Lead Time to Merge

### Limitações do notebook original

1. **Boxplot sem distribuição interna**: o boxplot suprime outliers (`showfliers=False`) e esconde
   a densidade real da distribuição — a mesma caixa pode esconder uma distribuição unimodal
   estreita ou uma bimodal larga.
2. **Três funções quase idênticas**: o notebook iterou três vezes sobre a mesma função
   (`plot_academic_log_boxplot`, `plot_academic_log_boxplot_only_reviewed`) sem consolidar.
3. **Apenas 7 repos**: os repos maiores foram omitidos.

### Abordagem aqui

- Violin plot com log₁₀ no eixo Y — preserva a forma da distribuição.
- Clipping no P99 para não deixar outliers extremos colapsarem o violino.
- Dois gráficos: (A) todos os PRs mergeados, (B) apenas PRs com pelo menos um revisor humano.
- Todos os 11 repositórios.

In [ ]:
# (A) All merged PRs with lead_time_hours > 0
groups_merged = {
    label: df[df['merged_at'].notna() & (df['lead_time_hours'] > 0)].copy()
    for label, df in groups.items()
}

_ = violin(
    groups_merged,
    metric='lead_time_hours',
    ylabel='Lead Time (hours)',
    title='Lead Time to Merge — All Merged PRs',
    log_scale=True,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'leadtime_all.pdf',
)

In [ ]:
# (B) Only PRs with ≥1 human reviewer
groups_reviewed = {}
for label, df in groups.items():
    df2 = df[df['merged_at'].notna() & (df['lead_time_hours'] > 0)].copy()
    df2['n_human_rev'] = df2['reviewers'].apply(count_human_reviewers)
    groups_reviewed[label] = df2[df2['n_human_rev'] >= 1]

_ = violin(
    groups_reviewed,
    metric='lead_time_hours',
    ylabel='Lead Time (hours)',
    title='Lead Time to Merge — PRs with ≥1 Human Reviewer',
    log_scale=True,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'leadtime_reviewed.pdf',
)

In [ ]:
# Descriptive statistics
rows = []
for label, df in groups_merged.items():
    vals = df['lead_time_hours'].dropna()
    rows.append({
        'Group':   label,
        'n':       len(vals),
        'Median':  round(vals.median(), 1),
        'IQR':     round(vals.quantile(0.75) - vals.quantile(0.25), 1),
        'P90 (h)': round(vals.quantile(0.90), 1),
        'P99 (h)': round(vals.quantile(0.99), 1),
    })
display(pd.DataFrame(rows).set_index('Group'))

---
## 2  PR Complexity — Comments and Commits

### Limitações do notebook original

O notebook original usava barras de **média** para comments e commits. Isso oculta:
- Quanto os projetos variam internamente (algumas PRs com dezenas de comments, outras com zero).
- Se a distribuição é unimodal ou bimodal.
- A influência de outliers na média.

Violin plots com escala log (necessária pois PRs sem nenhum comment são comuns)
permitem ver a densidade completa — incluindo o spike de PRs com 0 comentários.

In [ ]:
# Comments: include 0-comment PRs via symlog or shift by 1
# Strategy: add 0.5 before log so that 0 maps to log10(0.5) ≈ -0.3,
# making zero-comment PRs visible as a cluster near the bottom.
groups_comments = {
    label: df.assign(comments_plot=df['comments'].fillna(0) + 0.5)
    for label, df in groups.items()
}

_ = violin(
    groups_comments,
    metric='comments_plot',
    ylabel='Comments per PR',
    title='Comments per PR — Distribution by Repository',
    log_scale=True,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'complexity_comments.pdf',
)
print('Note: x-axis is log10(comments + 0.5). PRs with 0 comments cluster near log = −0.3.')

In [ ]:
_ = violin(
    groups,
    metric='commits',
    ylabel='Commits per PR',
    title='Commits per PR — Distribution by Repository',
    log_scale=True,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'complexity_commits.pdf',
)

In [ ]:
rows = []
for label, df in groups.items():
    c  = df['comments'].fillna(0)
    cm = df['commits'].dropna()
    rows.append({
        'Group':            label,
        'n':                len(df),
        'Comments median':  round(c.median(), 1),
        'Comments P75':     round(c.quantile(0.75), 1),
        'PRs 0 comments (%)': round((c == 0).mean() * 100, 1),
        'Commits median':   round(cm.median(), 1),
        'Commits P75':      round(cm.quantile(0.75), 1),
    })
display(pd.DataFrame(rows).set_index('Group'))

---
## 3  Review Process

O processo de revisão tem duas dimensões complementares:

1. **Coverage** — proporção de PRs que receberam pelo menos um revisor humano.
   Representada como barra horizontal (dado categórico/proporcional).
2. **Intensity** — quantos revisores humanos participaram, *entre as PRs revisadas*.
   Representada como violin (dado contínuo com distribuição).
3. **Time to first review** — horas entre a criação e o primeiro comentário/review.
   Representada como violin com escala log.

### Melhoria sobre o original

- O original usava duas barras de média separadas (reviewers totais e humanos).
- A cobertura não era calculada diretamente como %; era inferida por proxy.
- Aqui separamos explicitamente coverage (%) de intensity (distribuição).
- Todos os 11 repos são incluídos.

In [ ]:
# ── Review coverage: % of PRs with ≥1 human reviewer ─────────────────────────
coverage = {}
for label, df in groups.items():
    n_human = df['reviewers'].apply(count_human_reviewers)
    coverage[label] = (n_human >= 1).mean() * 100

# Sort descending
cov_sorted = sorted(coverage.items(), key=lambda x: x[1], reverse=True)
labels_cov, vals_cov = zip(*cov_sorted)
colors_cov = [GROUP_COLORS[l] for l in labels_cov]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(list(labels_cov), list(vals_cov), color=colors_cov, edgecolor='none')

# Color-coded value labels
for bar, val in zip(bars, vals_cov):
    color = '#2e7d32' if val >= 80 else ('#f57c00' if val >= 50 else '#c62828')
    ax.text(min(val + 1, 100), bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold', color=color)

ax.axvline(80, color='#2e7d32', linewidth=1.2, linestyle='--', alpha=0.7, label='80% reference')
ax.set_xlim(0, 108)
ax.set_xlabel('PRs with ≥1 Human Reviewer (%)', fontweight='bold')
ax.set_title('Review Coverage by Repository', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.legend(fontsize=9, frameon=False)

fig.savefig(FIGURES_DIR / 'review_coverage.pdf', bbox_inches='tight')
print('Saved: review_coverage.pdf')
plt.show()

In [ ]:
# ── Review intensity: distribution of human reviewer count ───────────────────
# Only among PRs that had at least 1 human reviewer
groups_intensity = {}
for label, df in groups.items():
    tmp = df.copy()
    tmp['n_human_rev'] = tmp['reviewers'].apply(count_human_reviewers)
    groups_intensity[label] = tmp[tmp['n_human_rev'] >= 1]

_ = violin(
    groups_intensity,
    metric='n_human_rev',
    ylabel='Human Reviewers per PR',
    title='Review Intensity — Human Reviewers per Reviewed PR',
    log_scale=False,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'review_intensity.pdf',
)

In [ ]:
# ── Time to first human response ─────────────────────────────────────────────
# Column: time_to_first_human_response_hours
groups_ttr = {
    label: df[
        df['time_to_first_human_response_hours'].notna() &
        (df['time_to_first_human_response_hours'] > 0)
    ].copy()
    for label, df in groups.items()
}

# Report how many PRs have this data
for label, df in groups_ttr.items():
    total = len(groups[label])
    n = len(df)
    print(f'  {label:<30} {n:>6}/{total} PRs have TTR data')

_ = violin(
    groups_ttr,
    metric='time_to_first_human_response_hours',
    ylabel='Time to First Response (hours)',
    title='Time to First Human Response — Distribution by Repository',
    log_scale=True,
    clip_pct=99,
    min_val=0,
    save_path=FIGURES_DIR / 'review_ttr.pdf',
)

In [ ]:
rows = []
for label, df in groups.items():
    n_hrev  = df['reviewers'].apply(count_human_reviewers)
    rev_prs = df[n_hrev >= 1]
    lt      = groups_merged[label]['lead_time_hours'].dropna()
    ttr     = groups_ttr[label]['time_to_first_human_response_hours'].dropna()
    rows.append({
        'Group':               label,
        'Coverage (%)':        round((n_hrev >= 1).mean() * 100, 1),
        'Reviewers (median)':  round(rev_prs['reviewers'].apply(count_human_reviewers).median(), 2) if len(rev_prs) else np.nan,
        'Lead time med (h)':   round(lt.median(), 1) if len(lt) else np.nan,
        'TTR med (h)':         round(ttr.median(), 1) if len(ttr) else np.nan,
    })
display(pd.DataFrame(rows).set_index('Group'))

---
## 4  Collective Ownership

**Definição:** um arquivo é considerado "compartilhado" se foi modificado por mais de um autor único
ao longo de toda a história de PRs do projeto. O collective ownership é medido por:

1. **Distribuição de autores por arquivo** — quantos autores únicos tocaram cada arquivo?
   Distribuição exibida como violin para cada repositório.
2. **% de arquivos compartilhados** — proporção de arquivos com >1 autor.
   Exibida como barra horizontal (proporcional).

### Melhoria sobre o original

O notebook original mostrava apenas a **média** de autores por arquivo (1 número por repo),
perdendo toda a estrutura da distribuição. Aqui exibimos a distribuição completa:
é possível ver se um projeto tem poucos arquivos muito compartilhados e muitos tocados por 1 autor,
vs. um projeto onde a maioria dos arquivos é co-desenvolvida por vários autores.

In [ ]:
def compute_file_ownership(df: pd.DataFrame) -> pd.Series:
    """Return a Series with the number of unique authors per file hash."""
    file_authors: dict[str, set] = {}
    for _, row in df.iterrows():
        author = row['author']
        fh = row.get('file_hashes', '')
        if pd.isna(fh) or str(fh).strip() == '' or pd.isna(author):
            continue
        for h in str(fh).split(','):
            h = h.strip()
            if h:
                file_authors.setdefault(h, set()).add(author)
    return pd.Series([len(v) for v in file_authors.values()], name='authors_per_file')


print('Computing file ownership (may take ~20s)...')
ownership: dict[str, pd.Series] = {}
for label, df in groups.items():
    s = compute_file_ownership(df)
    ownership[label] = s
    shared_pct = (s > 1).mean() * 100
    print(f'  {label:<30} {len(s):>6} files  |  avg authors/file: {s.mean():.2f}  |  shared: {shared_pct:.1f}%')

In [ ]:
# Build a long-format DataFrame from the ownership Series
long_own = pd.concat(
    [pd.DataFrame({'group': label, 'value': s.values}) for label, s in ownership.items()],
    ignore_index=True,
)

# Clip at P99 per group
p_clip = long_own.groupby('group')['value'].quantile(0.99)
long_own['disp'] = long_own.apply(lambda r: min(r['value'], p_clip[r['group']]), axis=1)

order_own = (
    long_own.groupby('group')['value']
    .median().sort_values().index.tolist()
)
palette_own = [GROUP_COLORS[g] for g in order_own]

fig, ax = plt.subplots(figsize=(12, 5))
sns.violinplot(
    data=long_own, x='group', y='disp',
    order=order_own, palette=palette_own,
    inner='box', bw_adjust=0.8, cut=0, linewidth=0.8, ax=ax,
)

ylim = ax.get_ylim()
for i, grp in enumerate(order_own):
    med = long_own.loc[long_own['group'] == grp, 'value'].median()
    ax.text(i, ylim[1] * 0.97, f'{med:.1f}',
            ha='center', va='top', fontsize=8, color='#333',
            bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.75, ec='none'))

ax.set_xlabel('')
ax.set_ylabel('Authors per File', fontweight='bold')
ax.set_title('Collective Ownership — Authors per File Distribution', fontweight='bold', pad=10)
ax.set_xticklabels(order_own, rotation=30, ha='right', fontsize=9)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Category bars
ylim2 = ax.get_ylim()
bar_y = ylim2[0] - (ylim2[1] - ylim2[0]) * 0.05
n = len(order_own)
for idxs, color in [
    ([i for i, g in enumerate(order_own) if 'Participativo' in g], BP_PRIMARY),
    ([i for i, g in enumerate(order_own) if 'Academic' in g],      ACAD_PRIMARY),
    ([i for i, g in enumerate(order_own) if 'Market' in g],        MKT_PRIMARY),
]:
    if not idxs:
        continue
    ax.axhline(bar_y, xmin=(min(idxs)+0.05)/n, xmax=(max(idxs)+0.95)/n,
               color=color, linewidth=4, solid_capstyle='butt', clip_on=False)

handles = [
    mpatches.Patch(color=BP_PRIMARY,   label='Government (BP)'),
    mpatches.Patch(color=ACAD_PRIMARY, label='Academic'),
    mpatches.Patch(color=MKT_PRIMARY,  label='Market'),
]
ax.legend(handles=handles, loc='upper right', frameon=True, framealpha=0.85,
          title='Category', title_fontsize=9)

ax.text(0.5, -0.22, 'Displayed up to 99th percentile per group. Annotation = median.',
        transform=ax.transAxes, ha='center', fontsize=8, color='gray', style='italic')

fig.savefig(FIGURES_DIR / 'ownership_violin.pdf', bbox_inches='tight')
print('Saved: ownership_violin.pdf')
plt.show()

In [ ]:
# % shared files — horizontal bar
shared = {label: (s > 1).mean() * 100 for label, s in ownership.items()}
shared_sorted = sorted(shared.items(), key=lambda x: x[1], reverse=True)
labels_sh, vals_sh = zip(*shared_sorted)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(list(labels_sh), list(vals_sh),
               color=[GROUP_COLORS[l] for l in labels_sh], edgecolor='none')

for bar, val in zip(bars, vals_sh):
    ax.text(min(val + 0.5, 100), bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9, fontweight='bold')

ax.set_xlim(0, 110)
ax.set_xlabel('Files Modified by >1 Author (%)', fontweight='bold')
ax.set_title('Collective Ownership — % Shared Files', fontweight='bold', pad=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.savefig(FIGURES_DIR / 'ownership_shared_pct.pdf', bbox_inches='tight')
print('Saved: ownership_shared_pct.pdf')
plt.show()

---
## 5  Statistical Tests

Violin plots são descritivos. Para suportar afirmações de que as diferenças observadas
**não são atribuíveis ao acaso**, realizamos:

1. **Kruskal-Wallis** (omnibus não-paramétrico): testa se ao menos um grupo difere dos demais.
   Adequado porque as distribuições são fortemente assimétricas.
2. **Mann-Whitney U** (pairwise): comparação par-a-par entre BP e cada outro grupo.
   Reportamos p-value corrigido por Bonferroni e **rank-biserial correlation** (r) como effect size
   (|r| < 0.1 = negligível, 0.1–0.3 = pequeno, 0.3–0.5 = médio, > 0.5 = grande).

In [ ]:
def rank_biserial_r(x: np.ndarray, y: np.ndarray) -> float:
    """Compute rank-biserial correlation as effect size for Mann-Whitney U."""
    u_stat, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    n1, n2 = len(x), len(y)
    return 1 - (2 * u_stat) / (n1 * n2)


def run_tests(groups: dict, metric: str, min_val: float = 0,
              label: str = '') -> pd.DataFrame:
    """
    Run Kruskal-Wallis + pairwise Mann-Whitney (BP vs each other group).
    Returns a DataFrame with results.
    """
    samples = {}
    for grp, df in groups.items():
        col = df[metric].dropna()
        col = col[col > min_val]
        if len(col) >= 5:
            samples[grp] = col.values

    if len(samples) < 2:
        print(f'{label}: not enough groups.')
        return pd.DataFrame()

    h, p_kw = stats.kruskal(*samples.values())
    print(f'\n{label}')
    print(f'  Kruskal-Wallis  H={h:.2f}  p={p_kw:.2e}  ({"significant" if p_kw < 0.05 else "not significant"})')

    bp_vals = samples.get('Brasil Participativo', np.array([]))
    if len(bp_vals) == 0:
        return pd.DataFrame()

    n_comparisons = len(samples) - 1
    rows = []
    for grp, vals in samples.items():
        if grp == 'Brasil Participativo':
            continue
        u, p_raw = stats.mannwhitneyu(bp_vals, vals, alternative='two-sided')
        p_bonf   = min(p_raw * n_comparisons, 1.0)
        r        = rank_biserial_r(bp_vals, vals)
        effect   = 'large' if abs(r) > 0.5 else ('medium' if abs(r) > 0.3 else ('small' if abs(r) > 0.1 else 'negligible'))
        rows.append({
            'Comparison':     f'BP vs {grp}',
            'U':              int(u),
            'p (raw)':        round(p_raw, 4),
            'p (Bonferroni)': round(p_bonf, 4),
            'Significant':    p_bonf < 0.05,
            'r (effect)':     round(r, 3),
            'Effect size':    effect,
        })

    result = pd.DataFrame(rows).set_index('Comparison')
    display(result)
    return result

In [ ]:
_ = run_tests(groups_merged, 'lead_time_hours', min_val=0, label='Lead Time to Merge')

In [ ]:
_ = run_tests(groups, 'comments', min_val=-1, label='Comments per PR')

In [ ]:
_ = run_tests(groups, 'commits', min_val=0, label='Commits per PR')

In [ ]:
_ = run_tests(groups_intensity, 'n_human_rev', min_val=0, label='Human Reviewers (reviewed PRs)')

In [ ]:
# For ownership we have Series, not DataFrames — adapt
groups_own_df = {label: pd.DataFrame({'authors_per_file': s}) for label, s in ownership.items()}
_ = run_tests(groups_own_df, 'authors_per_file', min_val=0, label='Authors per File (Ownership)')

In [ ]:
print('Generated figures:')
for f in sorted(FIGURES_DIR.glob('*.pdf')):
    kb = f.stat().st_size / 1024
    print(f'  {f.name:<45} {kb:6.1f} KB')